In [1]:
import os
import pandas as pd

# ============================================================
# Paths / settings
# ============================================================

data_dir = "/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/Combined"

studies = [1, 2, 3, 4, 5]

participant_col = "participant"


# ============================================================
# Compare files
# ============================================================

for study in studies:

    print("\n" + "=" * 70)
    print(f"STUDY {study}")
    print("=" * 70)

    explanation_path = os.path.join(
        data_dir,
        f"df_long_by_explanation_for_R-Study{study}.csv"
    )

    regular_path = os.path.join(
        data_dir,
        f"df_long_for_R-Study{study}.csv"
    )

    # --------------------------------------------------------
    # Check that both files exist
    # --------------------------------------------------------

    if not os.path.exists(explanation_path):
        print(f"Missing: {explanation_path}")
        continue

    if not os.path.exists(regular_path):
        print(f"Missing: {regular_path}")
        continue

    # --------------------------------------------------------
    # Load
    # --------------------------------------------------------

    df_explanation = pd.read_csv(explanation_path)
    df_regular = pd.read_csv(regular_path)

    # --------------------------------------------------------
    # Basic dimensions
    # --------------------------------------------------------

    print("\nROWS:")
    print(f"  by_explanation: {len(df_explanation)}")
    print(f"  regular:        {len(df_regular)}")

    if len(df_explanation) == len(df_regular):
        print("  ✓ Same number of rows")
    else:
        print(
            f"  ✗ ROW COUNT DIFFERENCE: "
            f"{len(df_explanation) - len(df_regular):+d}"
        )

    print("\nCOLUMNS:")
    print(f"  by_explanation: {len(df_explanation.columns)}")
    print(f"  regular:        {len(df_regular.columns)}")

    # --------------------------------------------------------
    # Compare column names
    # --------------------------------------------------------

    explanation_cols = set(df_explanation.columns)
    regular_cols = set(df_regular.columns)

    only_explanation = sorted(explanation_cols - regular_cols)
    only_regular = sorted(regular_cols - explanation_cols)
    common_cols = sorted(explanation_cols & regular_cols)

    print("\nColumns only in by_explanation:")
    print(only_explanation if only_explanation else "  None")

    print("\nColumns only in regular:")
    print(only_regular if only_regular else "  None")

    print(f"\nCommon columns ({len(common_cols)}):")
    print(common_cols)

    # --------------------------------------------------------
    # Participant comparison
    # --------------------------------------------------------

    if (
        participant_col in df_explanation.columns
        and participant_col in df_regular.columns
    ):

        p_explanation = set(
            df_explanation[participant_col].dropna().astype(str)
        )

        p_regular = set(
            df_regular[participant_col].dropna().astype(str)
        )

        print("\nPARTICIPANTS:")
        print(f"  by_explanation: {len(p_explanation)} unique")
        print(f"  regular:        {len(p_regular)} unique")

        missing_from_explanation = sorted(
            p_regular - p_explanation
        )

        missing_from_regular = sorted(
            p_explanation - p_regular
        )

        if not missing_from_explanation and not missing_from_regular:
            print("  ✓ Exact same participant IDs")
        else:
            print("  ✗ Participant discrepancy")

            print("\n  In regular but NOT by_explanation:")
            print(
                missing_from_explanation
                if missing_from_explanation else "  None"
            )

            print("\n  In by_explanation but NOT regular:")
            print(
                missing_from_regular
                if missing_from_regular else "  None"
            )

        # ----------------------------------------------------
        # Compare number of rows contributed by each participant
        # ----------------------------------------------------

        counts_explanation = (
            df_explanation[participant_col]
            .astype(str)
            .value_counts()
            .rename("by_explanation")
        )

        counts_regular = (
            df_regular[participant_col]
            .astype(str)
            .value_counts()
            .rename("regular")
        )

        participant_counts = pd.concat(
            [counts_explanation, counts_regular],
            axis=1
        ).fillna(0).astype(int)

        participant_counts["difference"] = (
            participant_counts["by_explanation"]
            - participant_counts["regular"]
        )

        count_problems = participant_counts[
            participant_counts["difference"] != 0
        ]

        print("\nROWS PER PARTICIPANT:")

        if count_problems.empty:
            print("  ✓ Every participant contributes the same number of rows")
        else:
            print("  ✗ Participants with different row counts:")
            print(count_problems)

    else:
        print(
            f"\nWARNING: '{participant_col}' was not found "
            "in both files."
        )

    # --------------------------------------------------------
    # Compare values in columns shared by both files
    # --------------------------------------------------------

    if len(df_explanation) == len(df_regular):

        differing_columns = []

        for col in common_cols:

            a = df_explanation[col].reset_index(drop=True)
            b = df_regular[col].reset_index(drop=True)

            # Treat NaN == NaN
            equal = a.eq(b) | (a.isna() & b.isna())

            n_different = (~equal).sum()

            if n_different > 0:
                differing_columns.append((col, n_different))

        print("\nVALUE DIFFERENCES IN COMMON COLUMNS:")

        if not differing_columns:
            print(
                "  ✓ All shared columns are identical row-by-row"
            )
        else:
            for col, n in differing_columns:
                print(f"  {col}: {n} differing rows")

    print()


STUDY 1

ROWS:
  by_explanation: 900
  regular:        900
  ✓ Same number of rows

COLUMNS:
  by_explanation: 19
  regular:        8

Columns only in by_explanation:
['causal_color', 'causal_shape', 'causal_tail', 'color_high', 'feature_explanation_group', 'had_causal_explanation_for_feature', 'has_causal_explanation', 'shape_discrete_slider.response', 'shape_high', 'tail_discrete_slider.response', 'task_explanation_group']

Columns only in regular:
  None

Common columns (8):
['feature_dimension', 'feature_importance', 'feature_relevance', 'irrelevant_dim', 'participant', 'relevant_dim_1', 'relevant_dim_2', 'task']

PARTICIPANTS:
  by_explanation: 150 unique
  regular:        300 unique
  ✗ Participant discrepancy

  In regular but NOT by_explanation:
['151', '152', '153', '154', '155', '156', '157', '158', '159', '160', '161', '162', '163', '164', '165', '166', '167', '168', '169', '170', '171', '172', '173', '174', '175', '176', '177', '178', '179', '180', '181', '182', '183', '18